# 03 — Traits de forme, alignement des offsets et index de règles

**Projet** : Extraction d'entités nommées par spaCy — le tagger appris, corroboré par des règles

## Objectifs pédagogiques

1. Calculer les traits d'une mention depuis sa **surface** (longueur, chiffres, séparateurs, casse) et lire ce qu'ils séparent.
1. Vérifier que les offsets du corpus tombent sur des **tokens** spaCy : ce que le modèle apprend dépend de cet alignement.
1. Construire l'index de règles **sur le train** et mesurer ce qu'il ne peut pas savoir : les surfaces réservées à l'évaluation.
1. Comprendre pourquoi un gazetteer n'est pas un modèle : il ne généralise pas.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.data.schemas import (  # noqa: E402
    ENTITY_LABELS,
    NAME_LABELS,
)
from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un corpus réduit (320 messages) et un budget d'époques
# réduit : l'exécution reste rapide, et les valeurs absolues d'un notebook ne sont pas celles de la
# référence publiée par le README.
NB_DOCUMENTS = 320
NB_EPOCHS = 3

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
            ],
        )
    )

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(MODEL_NODE.get("target", "label"))
ID_COLUMN = str(CONFIG.data.id_column or "msg_id")
SPLIT_COLUMN = str(CONFIG.data.group_column or "split")

# Les notebooks lisent **et écrivent** dans leur propre bac à sable : le corpus réduit et les
# artefacts des notebooks ne touchent ni `data/raw` ni `artifacts/`, donc ceux de `make train`
# restent ceux de la référence.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PROJECT_ROOT / "outputs" / "notebooks" / "data",
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Types d'entités   : {len(ENTITY_LABELS)} — {', '.join(ENTITY_LABELS)}")
print(f"Bac à sable       : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticEntityCorpusGenerator
from src.data.loaders import EntityCorpusLoader

LOADER = EntityCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=tuple(CONFIG.data.formats),
    validation_enabled=bool(CONFIG.data.validation.raw),
    lazy_validation=bool(CONFIG.data.validation.lazy),
)

if LOADER.documents_path.exists():
    DOCUMENTS, SPANS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis le bac à sable : {len(DOCUMENTS)} messages, graine {METADATA['seed']}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté **dans le bac à sable** pour que les notebooks suivants et les pipelines travaillent
    # sur exactement les mêmes fichiers.
    bundle = SyntheticEntityCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, SPANS, METADATA = bundle.documents, bundle.queries, dict(bundle.metadata)
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_annotations(SPANS)
    LOADER.save_metadata(METADATA)
    print(f"Bac à sable vide : corpus de {len(DOCUMENTS)} messages généré et persisté")

DOCUMENTS = DOCUMENTS.sort_values(ID_COLUMN).reset_index(drop=True)
SPANS = SPANS.sort_values([ID_COLUMN, "start", "end"]).reset_index(drop=True)

TRAIN_DOCUMENTS = LOADER.split("train", frame=DOCUMENTS)
VAL_DOCUMENTS = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION_DOCUMENTS = LOADER.split("calibration", frame=DOCUMENTS)
TEST_DOCUMENTS = LOADER.split("test", frame=DOCUMENTS)
TRAIN_SPANS = LOADER.split_annotations("train", documents=DOCUMENTS, spans=SPANS)
VAL_SPANS = LOADER.split_annotations("val", documents=DOCUMENTS, spans=SPANS)
TEST_SPANS = LOADER.split_annotations("test", documents=DOCUMENTS, spans=SPANS)

print(
    "messages par split :",
    {
        name: len(frame)
        for name, frame in (
            ("train", TRAIN_DOCUMENTS),
            ("val", VAL_DOCUMENTS),
            ("calibration", CALIBRATION_DOCUMENTS),
            ("test", TEST_DOCUMENTS),
        )
    },
)
print(
    "mentions par split :",
    {
        name: len(frame)
        for name, frame in (("train", TRAIN_SPANS), ("val", VAL_SPANS), ("test", TEST_SPANS))
    },
)

## 1. Les traits de forme d'une mention

Le modèle apprend ses propres représentations ; ces traits ne le nourrissent pas. Ils servent à **mesurer** ce que la forme d'une mention dit déjà — et donc à savoir quel score est mérité par une règle plutôt que par un modèle.

In [ ]:
from src.features.build_features import (
    MESSAGE_FEATURES,
    SPAN_FEATURES,
    build_message_features,
    build_span_features,
    describe_features,
    feature_summary,
)

DOCUMENTATION = describe_features()
display(
    pd.DataFrame(
        {
            "trait": list(SPAN_FEATURES),
            "définition": [DOCUMENTATION[name] for name in SPAN_FEATURES],
        }
    )
)

FEATURES = build_span_features(DOCUMENTS, SPANS)
display(FEATURES[list(SPAN_FEATURES)].describe().round(3))

In [ ]:
RESUME_TYPES = feature_summary(FEATURES)
display(RESUME_TYPES.round(3))
print("total des mentions par type :", int(RESUME_TYPES["n_mentions"].sum()), "=", len(SPANS))

In [ ]:
MESSAGES = build_message_features(DOCUMENTS, SPANS)
display(MESSAGES[list(MESSAGE_FEATURES)].describe().round(3))
print(
    "part moyenne du texte couverte par une annotation :",
    f"{MESSAGES['annotated_char_ratio'].mean():.1%}",
)
print(
    "densité moyenne (mentions pour 100 caractères) :",
    round(float(MESSAGES["mentions_per_100_chars"].mean()), 3),
)

**Ce qu'il faut retenir**

- Ces traits sont **calculés** à partir de la surface (`surface`, pas une colonne déclarée) : un corpus qui se contredirait serait détecté ici.
- `feature_summary` agrège par type : les types réguliers ont une longueur et une densité de chiffres stables, les types « nom » beaucoup moins.
- `annotated_char_ratio` rappelle qu'une entité est une petite partie du message : la tâche est un repérage dans du texte, pas un classement de texte.

## 2. La signature de forme, type par type

La signature réduit une surface à ses classes de caractères (`Aa-#` : un mot capitalisé, un séparateur, des chiffres). C'est la lecture la plus pauvre d'une mention — et celle qui dit si un type est **lisible** par une règle, et où cette règle s'arrête.

In [ ]:
from src.features.build_features import shape_separability

par_type = (
    FEATURES.groupby(["label", "signature"], observed=True)
    .size()
    .rename("mentions")
    .reset_index()
    .sort_values("mentions", ascending=False)
)
display(par_type.head(15))
print(
    "signatures distinctes par type :", FEATURES.groupby("label")["signature"].nunique().to_dict()
)

In [ ]:
SEPARABILITE = shape_separability(DOCUMENTS, SPANS)
print(f"signatures observées : {len(SEPARABILITE)}")
print(
    "signatures qui ne portent qu'un seul type :",
    int((SEPARABILITE["n_labels"] == 1).sum()),
    "/",
    len(SEPARABILITE),
)
display(SEPARABILITE.sort_values("n_spans", ascending=False).head(10).round(4))

**Ce qu'il faut retenir**

- Sur ce corpus, aucune signature n'est partagée par plusieurs types : la forme ne se trompe pas de **type** — elle ne sait pas **trouver** les bornes, et elle est muette sur une forme inédite.
- Les types « nom » (produit, transporteur) multiplient les signatures : leur forme ne dit rien de plus que « un mot » ou « deux mots capitalisés », ce qui rend la liste apprise utile — et bornée aux surfaces déjà vues.
- La pureté (`purity`) est la part des mentions d'une signature qui portent son type dominant : à 1,0, la forme est un indice sûr — jamais une extraction.

## 3. Tokénisation et alignement des offsets

Un pipeline spaCy travaille sur des **tokens**, le corpus est annoté en **caractères**. L'alignement est la jointure entre les deux : une mention qui ne tombe pas sur des frontières de tokens est ignorée à l'entraînement — et le projet publie le nombre de mentions ainsi perdues.

In [ ]:
import spacy

# Pipeline vide : tokénisation seule, aucun modèle pré-entraîné, aucun téléchargement.
nlp = spacy.blank("fr")

exemple = VAL_DOCUMENTS.iloc[0]
doc = nlp(str(exemple[TEXT_COLUMN]))
print("texte   :", str(exemple[TEXT_COLUMN])[:110], "...")
print("tokens  :", [token.text for token in doc][:16])
print()
for row in SPANS[SPANS["msg_id"] == exemple[ID_COLUMN]].itertuples(index=False):
    aligned = doc.char_span(int(row.start), int(row.end), label=str(row.label))
    rendu = aligned.text if aligned is not None else "DÉSALIGNÉ (aucun token exact)"
    print(f"{row.label!s:12s} {str(row.surface)!r:30s} -> {rendu}")

In [ ]:
TEXTS = dict(zip(DOCUMENTS[ID_COLUMN], DOCUMENTS[TEXT_COLUMN], strict=True))


def alignment_report(documents: pd.DataFrame, spans: pd.DataFrame) -> dict[str, float]:
    """Measure how many annotated mentions fall on token boundaries."""
    total = aligned = 0
    for message_id, groupe in spans.groupby(ID_COLUMN):
        doc = nlp(str(TEXTS[message_id]))
        for row in groupe.itertuples(index=False):
            total += 1
            if doc.char_span(int(row.start), int(row.end), label=str(row.label)) is not None:
                aligned += 1
    return {
        "mentions": total,
        "alignées": aligned,
        "désalignées": total - aligned,
        "taux d'alignement": round(aligned / max(total, 1), 4),
    }


ALIGNMENT = alignment_report(TRAIN_DOCUMENTS, TRAIN_SPANS)
print("alignement des annotations du train :", ALIGNMENT)

decalé = TRAIN_SPANS.copy()
decalé["start"] = decalé["start"] + 1
print("mêmes annotations décalées d'un caractère :", alignment_report(TRAIN_DOCUMENTS, decalé))

**Ce qu'il faut retenir**

- Le générateur écrit les annotations **au moment de l'écriture** du texte : l'alignement est donc parfait par construction, et il est mesuré pour le prouver.
- Un corpus réel arrive souvent désaligné ; c'est exactement ce que le taux d'alignement du rapport d'entraînement (`alignment_aligned_rate`) publie.
- Une mention désalignée n'est pas « presque » apprise : elle est ignorée, donc comptée comme manquée à l'évaluation.

## 4. L'index de règles, appris sur le train seulement

La couche de règles a deux étages : des **motifs** déclarés (commande, montant, date, transporteur) et un **index** des surfaces de noms vues à l'entraînement. Ce second étage est une liste : il ne peut pas reconnaître une surface qu'il n'a jamais lue.

In [ ]:
from src.models.rules import GazetteerIndex, rule_spans

INDEX = GazetteerIndex.from_spans(TRAIN_SPANS)
print(
    "surfaces indexées par type :",
    {label: len(surfaces) for label, surfaces in INDEX.surfaces.items()},
)

reserved = SPANS[SPANS["holdout"].astype(bool) & SPANS["label"].isin(NAME_LABELS)]
inconnues = sum(1 for surface in reserved["surface"] if str(surface).casefold() not in INDEX.lookup)
print(f"surfaces réservées de type « nom » : {len(reserved)}")
print(f"dont absentes de l'index (donc introuvables pour lui) : {inconnues}")

In [ ]:
exemple = DOCUMENTS[DOCUMENTS["split"] == "test"].iloc[0]
texte = str(exemple[TEXT_COLUMN])
print("texte :", texte[:120], "...")
print()
for span in rule_spans(texte, INDEX):
    print(
        f"{span.rule:20s} {span.label:12s} [{span.start}:{span.end}] "
        f"{texte[span.start : span.end]!r}"
    )

**Ce qu'il faut retenir**

- Chaque span de règle publie la règle qui l'a produit (`motif:date`, `index:produit`) : une prédiction doit toujours pouvoir s'expliquer.
- Les motifs généralisent (une date inédite reste une date), l'index non (un nom inédit n'est rien) : c'est la raison d'être des surfaces réservées.
- L'index est construit **après** le découpage : le construire sur le corpus entier serait une fuite de données, et les tests le vérifient.

## Synthèse

| Étage | Objet | Ce qu'il sait | Ce qu'il ignore |
| --- | --- | --- | --- |
| Traits de forme | `build_span_features` | longueur, chiffres, casse, séparateurs | le sens du texte |
| Tokénisation | `spacy.blank('fr')` | frontières de tokens, offsets | toute connaissance linguistique |
| Motifs | `rule_spans` (patterns) | les écritures régulières | les noms |
| Index | `GazetteerIndex.from_spans(TRAIN_SPANS)` | les surfaces vues à l'entraînement | les surfaces réservées |

**Suite** : `04_model_exploration.ipynb` compare les trois algorithmes qui utilisent ces étages.